**Building a labeled eval loop over a stratified dev subset** means sampling deliberately, not just grabbing the first N rows.
- The full dataset is roughly 40% FD / 30% Non-FD / 30% Multiple Category — a plain random sample inherits that imbalance and makes every metric look better than it really is on the harder classes
- **Stratified** means equal counts per class instead — `groupby("label").sample(n=10)` pulls exactly 10 of each, so all three classes get judged with equal weight
- A fixed `random_state` makes this the **same 30 emails every time** you re-run it — without that, you can't tell if a metric moved because the prompt changed or because the sample changed
- The loop wraps each call in `try/except` — one truncated/failed call shouldn't kill the whole eval run; it gets recorded as its own `"ERROR"` category instead of crashing or silently vanishing

**Precision, recall, and F1 for the FD class specifically** — not just an overall accuracy number — because FD is the business-critical class.
- **Recall** answers "of all the *real* FD emails, how many did we catch?" — a low FD recall means actual FD complaints are getting silently misrouted, which is the costly failure mode
- **Precision** answers "of everything we *labeled* FD, how many actually were?" — lower precision just means some Non-FD emails get extra (unnecessary) attention, a cheaper mistake
- That asymmetry is why your classical pipeline tracked **FD Recall** (0.97) as the headline number, not plain accuracy — and why the script below pulls it out separately from the per-class table

**A confusion matrix** turns "94% accurate" into "which mistakes are we actually making" — the same prediction can be right for the wrong reason if you only look at the aggregate.
- Rows are the **true** label, columns are the **predicted** label — the diagonal is everything correct, everything off-diagonal is a specific kind of mistake
- A 3-class problem (FD / Non-FD / Multiple Category) means **6 distinct ways to be wrong**, not just one "wrong" bucket — and they're not equally costly
- Printed as a small `pandas` table here rather than a chart — at dev-set scale (30 emails) a 3×3 grid of numbers is more useful than a visual, since you'll want to read exact counts, not eyeball intensity

**Comparing against the MuRIL baseline (0.97 FD Recall)** is the first real checkpoint — the first time a GenAI number gets to stand next to a number that's already deployed and trusted.
- `MURIL_BASELINE_FD_RECALL = 0.97` sits at the top of the script as a named constant, not a number you have to remember and retype into every comparison
- The point isn't to "win" on the very first prompt version — `v0` almost certainly won't beat 0.97, and that's fine; the point is having an honest number to track as `v1` → `v2` → `v3` close the gap
- Recall the project's own framing from the start: GenAI was scoped as **Layer 4**, handling the ~1–3% MuRIL is uncertain about — so the realistic target isn't "beat MuRIL overall," it's "be strong specifically on the hard slice MuRIL already struggles with"

**Deciding what to fix next by error type** — not the aggregate score — is what actually tells you which prompt edit to make.
- Two classifiers with identical 80% accuracy can be failing in completely different ways — one might be missing every `Multiple Category` email, the other might be confusing `FD` and `Non-FD` roughly evenly — and those need different fixes
- `error_breakdown()` groups every mistake by `(true_label, predicted_label)` pair and sorts by count — the **top row** of that table is where the next prompt-engineering effort should go, not wherever feels intuitive
- This is exactly the discipline your classical pipeline already used during MuRIL's hyperparameter sweep — react to *which* errors are happening, not just whether the number went up or down

In [1]:
import time
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
from dotenv import load_dotenv
from classifier_core import PROMPT_REGISTRY, classify_email
import anthropic
import os

load_dotenv()  # reads .env in the current working directory
api_key = os.getenv("anthropic_api_key")
client = anthropic.Anthropic(api_key=api_key)

MODEL_ID = "claude-haiku-4-5-20251001"
DATA_PATH = "../data/fd_dataset_messy.csv"
LABELS = ["FD", "Non-FD", "Multiple Category"]

In [2]:
# Your deployed classical pipeline's FD Recall (fine-tuned MuRIL). This is
# the number every GenAI version below has to be measured against, not just
# "is the accuracy good" in the abstract.
MURIL_BASELINE_FD_RECALL = 0.97
 
N_PER_CLASS = 10          # dev-set size per class -> 30 emails total, kept
                          # small on purpose for fast/cheap iteration
RANDOM_STATE = 42         # fixed seed -> same dev set every run, comparable
                          # results across versions and across days

In [3]:
# ----------------------------------------------------------------------
# Sub-topic 1: stratified dev subset + the eval loop itself
# ----------------------------------------------------------------------
def build_dev_subset(path: str, n_per_class: int, random_state: int) -> pd.DataFrame:
    """Stratified sample: equal emails per class, not a random slice of the
    whole file. The full dataset is ~40% FD / 30% Non-FD / 30% Multiple
    Category -- a plain random sample would inherit that imbalance and make
    every aggregate metric look better than it really is on the hard classes."""
    df = pd.read_csv(path)
    dev = (
        df.groupby("label", group_keys=False)
        .sample(n=n_per_class, random_state=random_state)
        .reset_index(drop=True)
    )
    return dev
 
 

In [4]:
def run_eval_loop(version: str, dev_set: pd.DataFrame, pause: float = 0.0) -> pd.DataFrame:
    """Classify every row in dev_set under one prompt version. Returns one
    row per email: true label, predicted label (or 'ERROR'), and the
    model's stated reason -- the raw material for every metric below."""
    rows = []
    for _, row in dev_set.iterrows():
        try:
            result = classify_email(version, row["subject"], row["content"])
            predicted = result.label
            reason = result.reason
        except Exception as e:
            # Don't let one truncated/failed call kill the whole eval run --
            # record it as its own category so the failure rate is visible
            # in the results instead of silently disappearing from them.
            predicted = "ERROR"
            reason = str(e)
        rows.append({
            "subject": row["subject"],
            "true_label": row["label"],
            "predicted_label": predicted,
            "model_reason": reason,
        })
        if pause:
            time.sleep(pause)
    return pd.DataFrame(rows)
 
 

In [5]:
# ----------------------------------------------------------------------
# Sub-topic 2: precision / recall / F1, with FD called out specifically --
# FD Recall is the business metric (a missed FD complaint is the costly
# error), not just an entry in a generic classification report.
# ----------------------------------------------------------------------
def compute_metrics(results: pd.DataFrame) -> dict:
    valid = results[results["predicted_label"] != "ERROR"]
    precision, recall, f1, _ = precision_recall_fscore_support(
        valid["true_label"], valid["predicted_label"], labels=LABELS, zero_division=0
    )
    per_class = {
        label: {"precision": p, "recall": r, "f1": f}
        for label, p, r, f in zip(LABELS, precision, recall, f1)
    }
    error_rate = (results["predicted_label"] == "ERROR").mean()
    return {"per_class": per_class, "fd_recall": per_class["FD"]["recall"], "error_rate": error_rate}
 

In [6]:
# ----------------------------------------------------------------------
# Sub-topic 3: confusion matrix -- GenAI predictions vs. ground truth
# ----------------------------------------------------------------------
def print_confusion_matrix(results: pd.DataFrame) -> pd.DataFrame:
    valid = results[results["predicted_label"] != "ERROR"]
    cm = confusion_matrix(valid["true_label"], valid["predicted_label"], labels=LABELS)
    cm_df = pd.DataFrame(cm, index=[f"true_{l}" for l in LABELS], columns=[f"pred_{l}" for l in LABELS])
    print(cm_df.to_string())
    return cm_df
 

In [7]:
# ----------------------------------------------------------------------
# Sub-topic 5: error-type breakdown -- which confusions actually dominate,
# instead of staring at one aggregate accuracy number
# ----------------------------------------------------------------------
def error_breakdown(results: pd.DataFrame) -> pd.DataFrame:
    errors = results[results["predicted_label"] != results["true_label"]]
    breakdown = (
        errors.groupby(["true_label", "predicted_label"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )
    return breakdown
 

In [8]:
if __name__ == "__main__":
    dev_set = build_dev_subset(DATA_PATH, N_PER_CLASS, RANDOM_STATE)
    print(f"Dev subset: {len(dev_set)} emails ({N_PER_CLASS} per class, seed={RANDOM_STATE})")
    print(dev_set["label"].value_counts().to_string())
 
    # Sub-topic 4: run every registered prompt version on the SAME dev set --
    # this is what Chapter 4's versioned registry was actually built for.
    summary_rows = []
    for version in PROMPT_REGISTRY:
        print("\n" + "=" * 70)
        print(f"VERSION: {version}")
        print("=" * 70)
 
        results = run_eval_loop(version, dev_set)
        metrics = compute_metrics(results)
 
        print("\nPer-class precision / recall / F1:")
        for label in LABELS:
            m = metrics["per_class"][label]
            print(f"  {label:18s} precision={m['precision']:.2f}  recall={m['recall']:.2f}  f1={m['f1']:.2f}")
 
        print(f"\nFailed/unparseable calls: {metrics['error_rate']:.1%}")
 
        print("\nConfusion matrix:")
        print_confusion_matrix(results)
 
        errs = error_breakdown(results)
        if not errs.empty:
            print("\nTop error types:")
            print(errs.to_string(index=False))
 
        # Checkpoint against the deployed MuRIL baseline
        fd_recall = metrics["fd_recall"]
        delta = fd_recall - MURIL_BASELINE_FD_RECALL
        sign = "+" if delta >= 0 else ""
        print(f"\nFD Recall: {fd_recall:.2f}  vs. MuRIL baseline: {MURIL_BASELINE_FD_RECALL:.2f}  ({sign}{delta:.2f})")
 
        summary_rows.append({
            "version": version,
            "fd_recall": fd_recall,
            "fd_precision": metrics["per_class"]["FD"]["precision"],
            "error_rate": metrics["error_rate"],
        })
 
    print("\n" + "=" * 70)
    print("SUMMARY -- every version, same dev set")
    print("=" * 70)
    print(pd.DataFrame(summary_rows).to_string(index=False))
 

Dev subset: 30 emails (10 per class, seed=42)
label
FD                   10
Multiple Category    10
Non-FD               10

VERSION: v0_zero_shot

Per-class precision / recall / F1:
  FD                 precision=0.64  recall=0.70  f1=0.67
  Non-FD             precision=0.62  recall=0.50  f1=0.56
  Multiple Category  precision=0.73  recall=0.80  f1=0.76

Failed/unparseable calls: 0.0%

Confusion matrix:
                        pred_FD  pred_Non-FD  pred_Multiple Category
true_FD                       7            3                       0
true_Non-FD                   2            5                       3
true_Multiple Category        2            0                       8

Top error types:
       true_label   predicted_label  count
               FD            Non-FD      3
           Non-FD Multiple Category      3
Multiple Category                FD      2
           Non-FD                FD      2

FD Recall: 0.70  vs. MuRIL baseline: 0.97  (-0.27)

VERSION: v1_domain_knowledge

